# 07 — H100 Video Inference Profiling (Stage 2)

**Purpose:** Profile inference latency and throughput on video input,
targeting real-time deployment requirements for the EcoCAR vehicle.

**Metrics:**
- Per-frame latency (ms) — model + NMS + lane postprocessing
- Throughput (FPS) at batch=1 and batch=N
- GPU memory usage
- Qualitative video overlay output

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm opencv-python-headless

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/EcoCAR-Perception-Pipeline-YOLO26-BDD100K/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [ ]:
import torch
import cv2
import numpy as np
import time
from lib.config import cfg
from lib.models import get_net
from lib.core.general import non_max_suppression, scale_coords
from lib.utils import show_seg_result, plot_one_box

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# ── Load model ──
cfg.defrost()
cfg.DRIVE.CHECKPOINT_DIR = '/content/drive/MyDrive/EcoCAR/checkpoints'
cfg.freeze()

model = get_net(cfg).to(device)
model.gr = 1.0
model.nc = 5
model.names = cfg.MODEL.VEHICLE_CLASSES

ckpt = torch.load(os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth'), map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()
model.half()  # FP16 for speed
print('Model loaded in FP16 mode')

In [ ]:
# ── Latency benchmark (synthetic input) ──
warmup_iters = 50
bench_iters = 200

dummy = torch.randn(1, 3, 640, 640, device=device, dtype=torch.float16)

# Warmup
for _ in range(warmup_iters):
    with torch.no_grad():
        _ = model(dummy)
torch.cuda.synchronize()

# Benchmark
latencies = []
for _ in range(bench_iters):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        det_out, ll_seg_out = model(dummy)
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
print(f'Latency (batch=1, FP16):')
print(f'  Mean:   {latencies.mean():.2f} ms')
print(f'  Median: {np.median(latencies):.2f} ms')
print(f'  P95:    {np.percentile(latencies, 95):.2f} ms')
print(f'  P99:    {np.percentile(latencies, 99):.2f} ms')
print(f'  FPS:    {1000/latencies.mean():.1f}')
print(f'  GPU Mem: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB')

In [ ]:
# ── Video inference demo ──
# Replace with your test video path
VIDEO_PATH = '/content/drive/MyDrive/EcoCAR/test_video.mp4'

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'Video: {w}x{h} @ {fps}fps, {total} frames')
    
    out_path = '/content/drive/MyDrive/EcoCAR/inference_output.mp4'
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    
    frame_times = []
    max_frames = min(total, 300)  # process up to 300 frames
    
    for fi in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Preprocess
        img = cv2.resize(frame, (640, 640))
        img_t = torch.from_numpy(img[:,:,::-1].copy()).permute(2,0,1).unsqueeze(0)
        img_t = img_t.half().to(device) / 255.0
        
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            det_out, ll_seg = model(img_t)
            inf_out, _ = det_out
            output = non_max_suppression(inf_out, conf_thres=0.25, iou_thres=0.45)
        torch.cuda.synchronize()
        frame_times.append((time.perf_counter() - t0) * 1000)
        
        # Draw results
        _, ll_mask = torch.max(ll_seg[0], 0)
        ll_mask = ll_mask.cpu().numpy().astype(np.uint8)
        ll_mask_resized = cv2.resize(ll_mask, (w, h))
        frame[ll_mask_resized > 0] = frame[ll_mask_resized > 0] * 0.5 + np.array([0, 255, 0]) * 0.5
        
        if len(output[0]):
            det = output[0].cpu()
            det[:, :4] = scale_coords((640, 640), det[:, :4], frame.shape).round()
            for *xyxy, conf, cls in det:
                label = f'{cfg.MODEL.VEHICLE_CLASSES[int(cls)]} {conf:.2f}'
                plot_one_box(xyxy, frame, label=label, line_thickness=2)
        
        writer.write(frame.astype(np.uint8))
    
    cap.release()
    writer.release()
    
    frame_times = np.array(frame_times)
    print(f'\nVideo inference ({len(frame_times)} frames):')
    print(f'  Mean latency: {frame_times.mean():.2f} ms ({1000/frame_times.mean():.1f} FPS)')
    print(f'  Output saved to {out_path}')
else:
    print(f'No test video found at {VIDEO_PATH}. Skipping video demo.')
    print('Place a test video there and re-run this cell.')